# CSET419 - Lab 11: Fine-Tuning GPT-2
## Component II: Recipe Instruction Generator (Food-Tech)

**Scenario:** You are an AI developer at a food-tech startup building a smart cooking
app. Your task is to fine-tune GPT-2 to generate step-by-step cooking instructions
when given a dish name as a prompt.

### Tasks
1. Reload a fresh GPT-2 model (independent of Component I)
2. Prepare the recipe instruction dataset and tokenize it
3. Fine-tune the model on recipe data
4. Generate recipe instructions using dish name prompts
5. Compare baseline vs. fine-tuned outputs

## Step 1: Setup and Load a Fresh GPT-2 Model

In [1]:
!pip install transformers datasets accelerate -q

In [2]:
import torch
import math
from transformers import (
    GPT2LMHeadModel, GPT2Tokenizer, Trainer,
    TrainingArguments, DataCollatorForLanguageModeling, set_seed
)
from datasets import Dataset

set_seed(42)

# Fresh instance - completely independent of Component I
tokenizer2 = GPT2Tokenizer.from_pretrained('gpt2')
model2 = GPT2LMHeadModel.from_pretrained('gpt2')
tokenizer2.pad_token = tokenizer2.eos_token
model2.config.pad_token_id = tokenizer2.eos_token_id

print(f'Model loaded. Parameters: {model2.num_parameters():,}')
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded. Parameters: 124,439,808
Device: cuda


## Step 2: Generate Baseline Recipes (Before Fine-Tuning)

In [3]:
def generate_text(model, tokenizer, prompt, max_length=100):
    model.eval()
    inputs = tokenizer.encode(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            inputs, max_length=max_length, temperature=0.8,
            top_k=50, do_sample=True, pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)

In [4]:
recipe_prompts = [
    'To make butter chicken',
    'For pasta carbonara',
    'To prepare a chocolate cake',
]

print('=== BASELINE RECIPES (Before Fine-Tuning) ===')
baseline2 = {}
for p in recipe_prompts:
    baseline2[p] = generate_text(model2, tokenizer2, p)
    print(f'Prompt: {p}\nOutput: {baseline2[p]}\n')

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


=== BASELINE RECIPES (Before Fine-Tuning) ===
Prompt: To make butter chicken
Output: To make butter chicken, just throw a little at a time. Add the chicken to the bottom of a large Dutch oven. Boil until it reaches 180 degrees. Stir in the milk and spices. Place chicken and chicken stock in a large saucepan. Bring to a boil, then allow to cool. Remove from heat. Reduce heat to medium-low. Cover with a lid and allow to cool. Cook in pan for 5 minutes, or until mixture is soft. Remove from pan, and remove

Prompt: For pasta carbonara
Output: For pasta carbonara:

1 cup fresh, fresh marinara, chopped

1/2 cup granulated sugar

1 tbsp all-purpose flour

1 tsp baking powder

1/2 tsp baking soda

1 tsp cayenne

1 tbsp olive oil

Method:

Preheat oven to 425 degrees f.

Combine chopped tomatoes, olive oil, flour and baking powder in a mixing bowl. Pour baking powder into the tomato

Prompt: To prepare a chocolate cake
Output: To prepare a chocolate cake, the top of the pan has to be the same 

## Step 3: Prepare Recipe Dataset and Fine-Tune

In [5]:
recipes = [
    'to make butter chicken start by marinating chicken pieces in yogurt with turmeric chili powder and garam masala for one hour.',
    'heat butter in a pan and fry onions until golden brown then add ginger garlic paste and cook for two minutes.',
    'add tomato puree and cook on low heat for ten minutes until the oil separates from the masala.',
    'add the marinated chicken and cook on medium heat for fifteen minutes until fully cooked.',
    'finish with fresh cream and kasuri methi and serve hot with naan or steamed rice.',
    'for pasta carbonara boil spaghetti in salted water until al dente and reserve half cup of pasta water.',
    'fry diced pancetta in olive oil until crispy and set aside.',
    'whisk together egg yolks parmesan cheese and black pepper in a bowl.',
    'toss the hot pasta with pancetta and remove from heat then quickly stir in the egg mixture.',
    'the residual heat will cook the eggs into a creamy sauce and serve immediately with extra parmesan.',
    'to prepare vegetable stir fry heat sesame oil in a wok on high heat.',
    'add sliced bell peppers broccoli florets and snap peas and toss for three minutes.',
    'pour in soy sauce oyster sauce and a pinch of sugar and stir well.',
    'add minced garlic and ginger and cook for one more minute until fragrant.',
    'serve the stir fry over steamed jasmine rice and garnish with sesame seeds.',
    'for chocolate chip cookies cream together butter and sugar until light and fluffy.',
    'beat in eggs one at a time then add vanilla extract and mix well.',
    'fold in flour baking soda and salt then gently stir in chocolate chips.',
    'scoop dough onto a baking sheet and bake at 180 degrees for twelve minutes until golden.',
    'let cookies cool on the tray for five minutes before transferring to a wire rack.',
]
print(f'Dataset size: {len(recipes)} recipe steps')

Dataset size: 20 recipe steps


In [6]:
dataset2 = Dataset.from_dict({'text': recipes})
tok2 = dataset2.map(
    lambda x: tokenizer2(x['text'], truncation=True, max_length=128, padding='max_length'),
    batched=True, remove_columns=['text']
)
split2 = tok2.train_test_split(test_size=0.15, seed=42)
print(f"Train: {len(split2['train'])} | Eval: {len(split2['test'])}")

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Train: 17 | Eval: 3


In [7]:
collator2 = DataCollatorForLanguageModeling(tokenizer=tokenizer2, mlm=False)

args2 = TrainingArguments(
    output_dir='./gpt2-recipes',
    num_train_epochs=15,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=50,
    eval_strategy='epoch',
    logging_steps=10,
    save_strategy='no',
    fp16=torch.cuda.is_available(),
)

trainer2 = Trainer(
    model=model2, args=args2,
    train_dataset=split2['train'], eval_dataset=split2['test'],
    data_collator=collator2
)

print('Starting fine-tuning on recipe data...')
trainer2.train()

Starting fine-tuning on recipe data...


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,No log,3.403592
2,3.926337,3.264174
3,3.926337,3.105294
4,3.443312,3.021800
5,3.443312,2.981154
6,2.728940,2.930787
7,2.728940,2.897742
8,2.090287,2.897580
9,2.090287,2.992910
10,1.577087,3.140069


TrainOutput(global_step=75, training_loss=2.0859599272410074, metrics={'train_runtime': 10.4889, 'train_samples_per_second': 24.311, 'train_steps_per_second': 7.15, 'total_flos': 16657367040000.0, 'train_loss': 2.0859599272410074, 'epoch': 15.0})

## Step 4: Evaluate and Compare Outputs

In [8]:
eval2 = trainer2.evaluate()
print(f'Perplexity: {math.exp(eval2["eval_loss"]):.2f}')

print('\n=== FINE-TUNED RECIPES (After Fine-Tuning) ===')
for p in recipe_prompts:
    ft = generate_text(model2, tokenizer2, p)
    print(f'Prompt: {p}')
    print(f'  Baseline:   {baseline2[p][:120]}')
    print(f'  Fine-Tuned: {ft[:120]}\n')

Perplexity: 36.75

=== FINE-TUNED RECIPES (After Fine-Tuning) ===
Prompt: To make butter chicken
  Baseline:   To make butter chicken, just throw a little at a time. Add the chicken to the bottom of a large Dutch oven. Boil until i
  Fine-Tuned: To make butter chicken add turmeric chili powder and garam masala and cook for one more minute until the chicken is soft

Prompt: For pasta carbonara
  Baseline:   For pasta carbonara:

1 cup fresh, fresh marinara, chopped

1/2 cup granulated sugar

1 tbsp all-purpose flour

1 tsp ba
  Fine-Tuned: For pasta carbonara boil spaghetti in salted water until al dente and reserve half cup of pasta water. Drain pasta and r

Prompt: To prepare a chocolate cake
  Baseline:   To prepare a chocolate cake, the top of the pan has to be the same size as the cake-like consistency.

I know the next t
  Fine-Tuned: To prepare a chocolate cake mix add flax eggs and vanilla extract. Turn off the heat and gently stir in the egg mixture.



## Expected Observations

- **Baseline** generates unrelated text (news / Wikipedia style) for cooking prompts
- **Fine-tuned** model generates step-by-step cooking instructions with
  ingredients, temperatures, and timing
- Generated recipes follow a realistic cooking flow: marinate -> cook -> serve
- Perplexity drops after fine-tuning

## Lightweight Model Alternatives

To swap the model, replace 'gpt2' with one of these in the from_pretrained() calls:

- distilgpt2 (82M) - Even smaller, trains faster
- EleutherAI/gpt-neo-125M (125M) - Open-source GPT-3 alternative
- TinyLlama/TinyLlama-1.1B-Chat-v1.0 (1.1B) - LLaMA architecture, efficient
- Qwen/Qwen2.5-0.5B (0.5B) - Alibaba small model, very capable
- facebook/opt-125m (125M) - Meta open GPT alternative